In [11]:
!pip install pyserial
import serial
import time
import pickle

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam

from keras.models import Sequential
from keras.layers import Dense

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import numpy as np

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

path_train = "gdrive/MyDrive/estencionistaBigData/training"
path_test = "gdrive/MyDrive/estencionistaBigData/test"
path_production = "gdrive/MyDrive/estencionistaBigData/production"

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [13]:
generator_training = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True)

In [14]:
generator_test = ImageDataGenerator(
    rescale=1./255 )

In [15]:
dados_training = generator_training.flow_from_directory(
    path_train,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary')

Found 100 images belonging to 2 classes.


In [16]:
dados_test = generator_test.flow_from_directory(
    path_test,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary')

Found 101 images belonging to 2 classes.


In [17]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
#new_model = model
#new_model.load_weights('gdrive/MyDrive/estencionistaBigData/model/new_2026.weights.h5')

In [18]:
# arquitetura da rede
print(model.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │     9,437,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,679,041 (36.92 MB)

 Trainable params: 9,679,041 (36.92 MB)

 Non-trainable params: 0 (0.00 B)

None


In [19]:

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [26]:
# @title
# Treinar o modelo
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=3)
historicoAprendizado = model.fit(dados_training, epochs=121, validation_data=dados_test, callbacks=[callback])

Epoch 1/121
4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.6100 - loss: 0.6894 - val_accuracy: 0.7525 - val_loss: 0.6852
Epoch 2/121
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.5700 - loss: 0.6852 - val_accuracy: 0.5050 - val_loss: 0.6789
Epoch 3/121
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 915ms/step - accuracy: 0.6500 - loss: 0.6587 - val_accuracy: 0.5050 - val_loss: 0.6746
Epoch 4/121
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.6400 - loss: 0.6559 - val_accuracy: 0.6040 - val_loss: 0.6558
Epoch 5/121
4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.6800 - loss: 0.6186 - val_accuracy: 0.7624 - val_loss: 0.6064
Epoch 6/121
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.6400 - loss: 0.6112 - val_accuracy: 0.5941 - val_loss: 0.6435
Epoch 7/121
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.7600 - loss: 0.5406 - val_accuracy: 0.7426 - val_loss: 0.5812
Epoch 8/121
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.6900 - loss: 0.5594 - val_accuracy: 0.7228 - val_loss: 0.5580
Epoch

In [24]:
model.save_weights('new_2037.weights.h5')

In [25]:
# Test value
precision = model.evaluate(dados_test)[1]
print("The accuracy of the model is:", precision)

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 348ms/step - accuracy: 0.5446 - loss: 0.6903
The accuracy of the model is: 0.5445544719696045


In [ ]:
# Test the model with a single image
caminhoImagem = "gdrive/MyDrive/estencionistaBigData/producao/V-01.jpeg"
imagemTeste = load_img(caminhoImagem, target_size=(224, 224))
imagemArray = img_to_array(imagemTeste)
imagemArray = np.expand_dims(imagemArray, axis=0)
imagemArray = imagemArray / 255.0

In [ ]:
imagemArray = img_to_array(imagemTeste)
imagemArray = np.expand_dims(imagemArray, axis=0)


In [ ]:
imagemArray = imagemArray / 255.0

In [ ]:

previsao = model.predict(imagemArray)

In [ ]:
if previsao[0][0] > 0.5:
    print("The image is classified as 'Sick'.")
    valor = "A"
else:
    print("The image is classified as 'Healthy'.")
    valor = "B"